In [1]:
import os
import json
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Paths
BASE_DIR = "/Users/bella/powertrust_rag"
CHROMA_DIR = os.path.join(BASE_DIR, "chroma_db")

# Load embedding model
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Connect to existing Chroma DB
vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
    collection_name="powertrust_malaysia"
)

print(f"Connected to Chroma DB")
print(f"Total documents: {vectorstore._collection.count()}")

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/14/9h283kqx58z105l23gk4_w740000gn/T/ipykernel_74067/762014043.py:19: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Connected to Chroma DB
Total documents: 6342


In [2]:
def query_rag(query, region=None, k=5):
    """
    Query RAG database and return results with similarity scores and sources.
    Returns: (results, scores, alerts, sources)
    """

    # Build filter
    where_filter = {"status": "active"}
    if region:
        where_filter["region"] = {"$contains": region}

    # Semantic search with scores
    try:
        results_with_scores = vectorstore.similarity_search_with_score(
            query=query,
            k=k,
            filter=where_filter
        )
    except Exception:
        results_with_scores = vectorstore.similarity_search_with_score(
            query=query,
            k=k
        )

    results = [doc for doc, score in results_with_scores]
    scores  = [score for doc, score in results_with_scores]

    # Proactive alert check
    alerts = []
    try:
        alert_results = vectorstore.similarity_search(query=query, k=50)
        for doc in alert_results:
            if doc.metadata.get("unknown_unknown_flag") == "True":
                alert = doc.metadata.get("alert_message", "")
                if alert and alert not in alerts:
                    alerts.append(alert)
    except Exception:
        pass

    # Format sources (deduplicated)
    seen = set()
    sources = []
    for doc in results:
        filename = doc.metadata.get("filename", "")
        title    = doc.metadata.get("title", "Unknown")
        org      = doc.metadata.get("organization", "Unknown")
        if filename and filename not in seen:
            seen.add(filename)
            sources.append({
                "title":    title,
                "org":      org,
                "filename": filename
            })

    return results, scores, alerts, sources


# Test
results, scores, alerts, sources = query_rag(
    "What is the LCOE for solar in Malaysia?",
    region="peninsular"
)

print(f"Results: {len(results)}")
print(f"Top score: {scores[0]:.4f}")
print(f"Alerts: {len(alerts)}")
print(f"\nSources found:")
for s in sources:
    print(f"  • {s['title']} — {s['org']}")
    print(f"    ({s['filename']})")

Results: 5
Top score: 0.1926
Alerts: 0

Sources found:
  • Solar ATAP Technical Assessment Study Process Flow for CCC and CAS — SEDA Malaysia / Energy Commission (ST)
    (seda_atap_ccc_cas_process_flow.pdf)
  • Solar ATAP Non-Domestic Technical Assessment Requirements Table — SEDA Malaysia / Energy Commission (ST)
    (non_domestic_application_process.png)
  • Guidelines of Solar Accelerated Transition Action Programme (Solar ATAP) — SEDA Malaysia / Energy Commission (ST)
    (seda_atap_guidelines_2025.pdf)
  • Solar ATAP Technical Assessment Study Application Form — SEDA Malaysia / Energy Commission (ST)
    (atap_assessment_study_form.pdf)
  • Malaysia: A Techno-Economic Analysis of Power Generation — BloombergNEF
    (bnef_malaysia_techno_economic_apr2025.pdf)


In [5]:
from ddgs import DDGS
# Time-sensitive keywords that trigger web search
TIME_KEYWORDS = [
    "latest", "current", "recent", "2025", "2026",
    "now", "today", "update", "new", "announce"
]

def needs_web_search(query, scores, threshold=0.5):
    """
    Determine if web search is needed.
    Triggers if:
    - Top RAG score is too high (low similarity), OR
    - Query contains time-sensitive keywords
    """
    top_score = min(scores) if scores else 999

    # Check time keywords
    query_lower = query.lower()
    has_time_keyword = any(kw in query_lower for kw in TIME_KEYWORDS)

    triggered_by_score   = top_score > threshold
    triggered_by_keyword = has_time_keyword

    return triggered_by_score or triggered_by_keyword, {
        "top_score":            round(top_score, 4),
        "threshold":            threshold,
        "triggered_by_score":   triggered_by_score,
        "triggered_by_keyword": triggered_by_keyword,
        "matched_keyword":      next((kw for kw in TIME_KEYWORDS if kw in query_lower), None)
    }


def search_web(query, max_results=3):
    """
    Search the web using DuckDuckGo.
    Appends 'Malaysia solar' to keep results relevant.
    Returns list of {title, url, snippet}
    """
    search_query = f"Malaysia solar {query}"
    results = []

    try:
        with DDGS() as ddgs:
            for r in ddgs.text(search_query, max_results=max_results):
                results.append({
                    "title":   r.get("title", ""),
                    "url":     r.get("href", ""),
                    "snippet": r.get("body", "")[:300]
                })
    except Exception as e:
        print(f"  ⚠️  Web search failed: {e}")

    return results


# Test
query = "latest solar policy update Malaysia 2026"
triggered, reason = needs_web_search(query, scores)

print(f"Query: {query}")
print(f"Trigger web search: {triggered}")
print(f"Reason: {reason}")

if triggered:
    print("\nRunning web search...")
    web_results = search_web(query)
    for r in web_results:
        print(f"\n  • {r['title']}")
        print(f"    {r['url']}")
        print(f"    {r['snippet'][:150]}")

Query: latest solar policy update Malaysia 2026
Trigger web search: True
Reason: {'top_score': 0.1926, 'threshold': 0.5, 'triggered_by_score': False, 'triggered_by_keyword': True, 'matched_keyword': 'latest'}

Running web search...

  • Solar ATAP Update 2026: What You Need to Know About Malaysia’s New Solar ATAP Program
    https://www.solarsunyield.com/latestnews/nid/173471/
    December 19, 2025 -... The Ministry of Energy Transition and Water Transformation (PETRA) has officially released an updated media statement today ann

  • Malaysia: 2026 Updates to Renewable Energy Schemes | Insight | Baker McKenzie
    https://www.bakermckenzie.com/en/insight/publications/2026/01/malaysia-2026-updates-to-renewable-energy-schemes
    January 28, 2026 -The 2026 updates include thenew Solar ATAP scheme, updates to the CRESS guidelines and updates regarding the standby charge payable 

  • Malaysia’s solar capacity surpasses 5.7 GW
    https://www.pv-magazine.com/2026/04/03/malaysias-solar-capa

In [6]:
from groq import Groq

# Initialise Groq client
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")



groq_client = Groq(api_key=GROQ_API_KEY)

def format_rag_context(results):
    """Format RAG chunks into a single context string for the LLM"""
    context_parts = []
    for i, doc in enumerate(results):
        context_parts.append(f"[Document {i+1}]\n{doc.page_content[:800]}")
    return "\n\n".join(context_parts)

def format_web_context(web_results):
    """Format web search results into context string"""
    context_parts = []
    for i, r in enumerate(web_results):
        context_parts.append(
            f"[Web Result {i+1}]\n"
            f"Title: {r['title']}\n"
            f"URL: {r['url']}\n"
            f"Content: {r['snippet']}"
        )
    return "\n\n".join(context_parts)

def format_sources_output(rag_sources, web_results=None):
    """Format sources section for final response"""
    lines = ["[SOURCES]"]

    if rag_sources:
        lines.append("Database:")
        for s in rag_sources:
            lines.append(f"  • {s['title']} — {s['org']}")
            lines.append(f"    ({s['filename']})")

    if web_results:
        lines.append("Web Search:")
        for r in web_results:
            lines.append(f"  • {r['title']}")
            lines.append(f"    {r['url']}")

    return "\n".join(lines)

def build_prompt(query, rag_context, web_context=None, region=None):
    """Build the LLM prompt combining RAG and web context"""

    region_note = f"The user is asking about {region} Malaysia." if region else ""

    web_section = ""
    if web_context:
        web_section = f"""
SUPPLEMENTARY WEB SEARCH RESULTS (use only if database is insufficient):
{web_context}
"""

    prompt = f"""You are an expert analyst for PowerTrust, helping developers and investors understand distributed solar development in Malaysia.

{region_note}

Answer the user's question using ONLY the context provided below. 
- Prioritise information from the DATABASE CONTEXT over web search results.
- If the database has sufficient information, do not rely on web results.
- If data is missing or unclear, explicitly say so — do not guess or fabricate.
- Be concise and specific. Use numbers and facts where available.

DATABASE CONTEXT:
{rag_context}
{web_section}

USER QUESTION: {query}

Provide a clear, structured answer. Do not repeat the sources — they will be appended separately."""

    return prompt


print("LLM helper functions defined")

✅ LLM helper functions defined


In [7]:
def run_agent(query, region=None):
    """
    Main hybrid agent function.
    1. Query RAG database
    2. Decide if web search is needed
    3. Call LLM with combined context
    4. Return structured response
    """

    print(f"Query: {query}")
    print(f"Region: {region or 'all'}\n")

    # Step 1: RAG query
    print("Step 1: Querying RAG database...")
    results, scores, alerts, rag_sources = query_rag(query, region=region, k=5)
    print(f"  → {len(results)} chunks found, top score: {min(scores):.4f}")

    # Step 2: Decide if web search needed
    print("Step 2: Checking if web search needed...")
    web_needed, reason = needs_web_search(query, scores)
    print(f"  → Web search triggered: {web_needed}")
    if web_needed:
        trigger = reason.get("matched_keyword") or f"low similarity ({reason['top_score']})"
        print(f"  → Reason: {trigger}")

    # Step 3: Web search if needed
    web_results = []
    if web_needed:
        print("Step 3: Running web search...")
        web_results = search_web(query, max_results=3)
        print(f"  → {len(web_results)} web results found")
    else:
        print("Step 3: Skipped (database sufficient)")

    # Step 4: Build prompt and call LLM
    print("Step 4: Generating answer...")
    rag_context = format_rag_context(results)
    web_context = format_web_context(web_results) if web_results else None

    prompt = build_prompt(query, rag_context, web_context, region)

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=1024
    )

    answer = response.choices[0].message.content

    # Step 5: Format final output
    sources_output = format_sources_output(rag_sources, web_results if web_results else None)

    alerts_output = ""
    if alerts:
        alerts_output = "\n[ALERTS]\n" + "\n".join([f"⚠️  {a}" for a in alerts])

    final_response = f"{answer}\n\n{sources_output}{alerts_output}"

    print("\n" + "="*60)
    print(final_response)
    print("="*60)

    return final_response


print("Agent function defined")

Agent function defined


In [8]:
# Test 1: Database-only question
print("\n" + "="*60)
print("TEST 1: Database-only question")
print("="*60)
run_agent(
    query="What is the CAPEX for a 1MW solar farm in Malaysia?",
    region="peninsular"
)


TEST 1: Database-only question
Query: What is the CAPEX for a 1MW solar farm in Malaysia?
Region: peninsular

Step 1: Querying RAG database...
  → 5 chunks found, top score: 0.1841
Step 2: Checking if web search needed...
  → Web search triggered: False
Step 3: Skipped (database sufficient)
Step 4: Generating answer...

**CAPEX for a 1MW Solar Farm in Peninsular Malaysia:**
The CAPEX for a 1MW solar farm in Peninsular Malaysia is approximately RM1.8 million. This is based on real-world pricing from a registered Malaysian solar installer, which includes labour, equipment, and margin. The cost per kWp for a 1MWp system is around RM1,800/kWp.

[SOURCES]
Database:
  • Solar Panel Installation Cost in Malaysia — Commercial CAPEX Benchmark — Progressture Solar
    (progressture_capex_2024.html)
  • Guidelines of Solar Accelerated Transition Action Programme (Solar ATAP) — SEDA Malaysia / Energy Commission (ST)
    (seda_atap_guidelines_2025.pdf)
  • Solar ATAP Non-Domestic Technical Assessm

"**CAPEX for a 1MW Solar Farm in Peninsular Malaysia:**\nThe CAPEX for a 1MW solar farm in Peninsular Malaysia is approximately RM1.8 million. This is based on real-world pricing from a registered Malaysian solar installer, which includes labour, equipment, and margin. The cost per kWp for a 1MWp system is around RM1,800/kWp.\n\n[SOURCES]\nDatabase:\n  • Solar Panel Installation Cost in Malaysia — Commercial CAPEX Benchmark — Progressture Solar\n    (progressture_capex_2024.html)\n  • Guidelines of Solar Accelerated Transition Action Programme (Solar ATAP) — SEDA Malaysia / Energy Commission (ST)\n    (seda_atap_guidelines_2025.pdf)\n  • Solar ATAP Non-Domestic Technical Assessment Requirements Table — SEDA Malaysia / Energy Commission (ST)\n    (non_domestic_application_process.png)\n  • Malaysia: A Techno-Economic Analysis of Power Generation — BloombergNEF\n    (bnef_malaysia_techno_economic_apr2025.pdf)\n[ALERTS]\n⚠️  Did you know? Data centers have reserved 43% of TNB's total grid

In [9]:
# Test 2: Time-sensitive question (should trigger web search)
print("\n" + "="*60)
print("TEST 2: Time-sensitive question")
print("="*60)
run_agent(
    query="What are the latest updates to Malaysia solar policy in 2026?",
    region="peninsular"
)


TEST 2: Time-sensitive question
Query: What are the latest updates to Malaysia solar policy in 2026?
Region: peninsular

Step 1: Querying RAG database...
  → 5 chunks found, top score: 0.1841
Step 2: Checking if web search needed...
  → Web search triggered: True
  → Reason: latest
Step 3: Running web search...
  → 3 web results found
Step 4: Generating answer...

**Latest Updates to Malaysia Solar Policy in 2026:**

1. **Solar ATAP Programme**: Replaces all NEM programmes, effective January 2026, with key improvements including:
	* No quota restrictions
	* Continuous programme with no predetermined end date
	* Higher capacity limits
	* SMP-based export credit for non-domestic users
	* Extended to agricultural users
2. **Replacement of NEM Programmes**: NEM programmes ended in June 2025, and Solar ATAP is the new programme in place.
3. **Regional Applicability**: Solar ATAP is applicable in Peninsular Malaysia, while Sarawak and Sabah operate under separate independent regulatory fram

"**Latest Updates to Malaysia Solar Policy in 2026:**\n\n1. **Solar ATAP Programme**: Replaces all NEM programmes, effective January 2026, with key improvements including:\n\t* No quota restrictions\n\t* Continuous programme with no predetermined end date\n\t* Higher capacity limits\n\t* SMP-based export credit for non-domestic users\n\t* Extended to agricultural users\n2. **Replacement of NEM Programmes**: NEM programmes ended in June 2025, and Solar ATAP is the new programme in place.\n3. **Regional Applicability**: Solar ATAP is applicable in Peninsular Malaysia, while Sarawak and Sabah operate under separate independent regulatory frameworks.\n\n**Note:** The information provided is based on the available data and may not be exhaustive. If additional details are required, further research may be necessary.\n\n[SOURCES]\nDatabase:\n  • National Energy Transition Roadmap (NETR) Part 2 — Roadmap in Full — Ministry of Economy, Malaysia\n    (netr_part2_2023.pdf)\n  • IEA-PVPS Malaysia 

In [10]:
# Test 3: Question with no good answer in database (low similarity)
print("\n" + "="*60)
print("TEST 3: Low similarity question")
print("="*60)
run_agent(
    query="What is the average grid connection waiting time in Sabah?",
    region="sabah"
)


TEST 3: Low similarity question
Query: What is the average grid connection waiting time in Sabah?
Region: sabah

Step 1: Querying RAG database...
  → 5 chunks found, top score: 0.1968
Step 2: Checking if web search needed...
  → Web search triggered: False
Step 3: Skipped (database sufficient)
Step 4: Generating answer...

The average grid connection waiting time in Sabah is not explicitly stated in the provided documents. However, Document 4 mentions that the SAIDI (System Average Interruption Duration Index) for Peninsular Malaysia is 47.88 minutes, and it is compared to Sabah's SAIDI of 203-286 minutes. 

It is essential to note that SAIDI measures the average duration of power interruptions, not the waiting time for grid connections. The actual waiting time for grid connections in Sabah is not available in the provided documents.

[SOURCES]
Database:
  • TNB Investor Presentation — December 2025 — Tenaga Nasional Berhad (TNB)
    (tnb_investor_deck_2025.pdf)
  • Electricity Sector

"The average grid connection waiting time in Sabah is not explicitly stated in the provided documents. However, Document 4 mentions that the SAIDI (System Average Interruption Duration Index) for Peninsular Malaysia is 47.88 minutes, and it is compared to Sabah's SAIDI of 203-286 minutes. \n\nIt is essential to note that SAIDI measures the average duration of power interruptions, not the waiting time for grid connections. The actual waiting time for grid connections in Sabah is not available in the provided documents.\n\n[SOURCES]\nDatabase:\n  • TNB Investor Presentation — December 2025 — Tenaga Nasional Berhad (TNB)\n    (tnb_investor_deck_2025.pdf)\n  • Electricity Sector in Malaysia — Wikipedia — Wikipedia\n    (wikipedia_malaysia_electricity.html)\n  • Technical Guideline for Interconnection of Distributed Generator to Distribution System — Tenaga Nasional Berhad (TNB)\n    (tnb_dg_interconnection_guideline_2018.pdf)\n  • Transition Roadmap: Malaysia's Efforts to Modernise its Pow

In [11]:
def ask(question, region=None):
    """
    Clean interface for the UI team.
    
    Parameters:
        question (str): User's question in natural language
        region (str):   'peninsular', 'sabah', 'sarawak', or None for all regions
    
    Returns:
        dict: {
            'answer':      str,
            'rag_sources': list of {title, org, filename},
            'web_sources': list of {title, url} or [],
            'alerts':      list of str or [],
            'web_used':    bool
        }
    """

    # RAG query
    results, scores, alerts, rag_sources = query_rag(question, region=region, k=5)

    # Web search decision
    web_needed, _ = needs_web_search(question, scores)
    web_results = search_web(question) if web_needed else []

    # Build prompt and call LLM
    rag_context = format_rag_context(results)
    web_context = format_web_context(web_results) if web_results else None
    prompt      = build_prompt(question, rag_context, web_context, region)

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=1024
    )

    answer = response.choices[0].message.content

    return {
        "answer":      answer,
        "rag_sources": rag_sources,
        "web_sources": [{"title": r["title"], "url": r["url"]} for r in web_results],
        "alerts":      alerts,
        "web_used":    web_needed
    }


# Quick test
result = ask("What incentives are available for solar in Malaysia?", region="peninsular")

print(f"Answer:\n{result['answer']}")
print(f"\nRAG Sources: {len(result['rag_sources'])}")
print(f"Web Sources: {len(result['web_sources'])}")
print(f"Alerts:      {len(result['alerts'])}")
print(f"Web used:    {result['web_used']}")

Answer:
**Incentives for Solar in Malaysia:**

1. **Export Credit Mechanism**: System Marginal Price (SMP) is available for excess energy exported to the grid.
2. **Power Purchase Agreements (PPA)**: Available with a tenure of 21 years, with TNB/SESB as the counterparty.
3. **Cost-saving opportunities**: Various renewable energy incentives are available for businesses and individuals.
4. **No quota limitation**: For non-domestic solar installations, there is no quota limit, up to 100% of maximum demand.
5. **Fee structures**: Different fee structures apply for domestic and non-domestic installations, with varying capacity thresholds (e.g., CAS and PSS fees).

Note: The specific details of these incentives may vary depending on the program (e.g., LSS, Solar ATAP) and region (e.g., peninsular Malaysia).

RAG Sources: 5
Web Sources: 0
Alerts:      0
Web used:    False
